## AgentCore Evaluations - avaliação online para agente LangGraph

Neste tutorial, você aprenderá a usar a avaliação online do AgentCore Evaluations aplicada a um agente LangGraph.

Para executar este laboratório, você deve primeiro ter criado o agente LangGraph usando o código na pasta [00-prereqs](../../00-prereqs) e criado seu avaliador personalizado usando o código em [01-creating-custom-evaluators](../../01-creating-custom-evaluators)

### O que você aprenderá
- Como executar avaliações online em um trace usando o AgentCore Starter toolkit

### Detalhes do tutorial

| Informação              | Detalhes                                                                              |
|:------------------------|:--------------------------------------------------------------------------------------|
| Tipo do tutorial        | Avaliando agente LangGraph com avaliadores online (integrados e personalizados)        |
| Componentes do tutorial | Configurando avaliação automatizada com avaliadores integrados e personalizados        |
| Vertical do tutorial    | Cross-vertical                                                                         |
| Complexidade do exemplo | Fácil                                                                                  |
| SDK utilizado           | Amazon Bedrock AgentCore Starter toolkit                                               |

### Avaliação online

A avaliação online permite o monitoramento de qualidade do tráfego em tempo real de agentes implantados. Diferente da avaliação sob demanda, que analisa interações específicas selecionadas, a avaliação online avalia continuamente o desempenho do seu agente em ambientes de produção com base no tráfego em tempo real.

A avaliação online consiste em três componentes principais. Primeiro, a **amostragem e filtragem de sessões** permite configurar regras específicas para avaliar interações do agente. Você pode definir amostragem baseada em porcentagem para avaliar uma parte de todas as sessões (por exemplo, 10%) ou definir filtros condicionais para avaliação mais direcionada. Segundo, você pode escolher entre **múltiplos métodos de avaliação**, incluindo criar novos avaliadores personalizados, usar avaliadores personalizados existentes ou selecionar entre avaliadores integrados. Por fim, as capacidades de **monitoramento e análise** permitem visualizar pontuações agregadas em dashboards, acompanhar tendências de qualidade ao longo do tempo, investigar sessões com pontuações baixas e analisar fluxos completos de interação da entrada à saída.

Com a avaliação online, você configura o sistema para monitorar automaticamente fontes de dados específicas — seja grupos de logs do CloudWatch contendo traces do agente ou endpoints do AgentCore Runtime. O serviço processa continuamente os traces recebidos com base nas suas regras de amostragem e filtragem, aplica os avaliadores escolhidos em tempo real e gera resultados detalhados no CloudWatch para análise. Este tipo de avaliação é particularmente útil para monitoramento em produção, detecção precoce de regressões de qualidade, identificação de padrões nas interações dos usuários e manutenção de desempenho consistente do agente em escala.

Uma vez que você cria e habilita uma configuração de avaliação online, o serviço executa continuamente em segundo plano, avaliando sessões conforme elas ocorrem e fornecendo visibilidade contínua das métricas de qualidade do seu agente. Você pode pausar, modificar ou excluir configurações a qualquer momento para adaptar sua estratégia de avaliação conforme suas necessidades evoluem.

### Gerando traces no AgentCore Observability a partir de um agente

O AgentCore Observability fornece visibilidade abrangente do comportamento do agente durante invocações, aproveitando traces [OpenTelemetry (OTEL)](https://opentelemetry.io/) como base para capturar e estruturar dados detalhados de execução. O AgentCore utiliza o [AWS Distro for OpenTelemetry (ADOT)](https://aws-otel.github.io/) para instrumentar diferentes tipos de traces OTEL em vários frameworks de agentes.

Quando seu agente está hospedado no AgentCore Runtime (como nosso agente neste tutorial), a instrumentação do AgentCore Observability é automática, com configuração mínima. Tudo o que você precisa fazer é incluir `aws-opentelemetry-distro` no `requirements.txt` e o AgentCore Runtime cuida da configuração OTEL automaticamente. Quando seu agente não está executando no AgentCore Runtime, você precisará instrumentá-lo com ADOT para que ele esteja disponível no AgentCore Observability. Você precisa configurar variáveis de ambiente para direcionar os dados de telemetria ao CloudWatch e executar seu agente com instrumentação OpenTelemetry.

O processo funciona da seguinte forma:

![session_traces](../../images/observability_traces.png)

Uma vez que os traces da sua sessão estejam disponíveis no AgentCore Observability, você pode usar o AgentCore Evaluations para avaliar o comportamento do seu agente. Para avaliações online, você não precisa fazer nada extra. Basta monitorar o desempenho do seu agente nos dashboards em tempo real.

### Como a avaliação online funciona com os traces

Na avaliação online, seu agente é invocado e gera traces no AgentCore Observability. Esses traces são mapeados para sessões e seus logs são disponibilizados em grupos de logs do Amazon CloudWatch. Com a avaliação online, um desenvolvedor cria uma configuração de avaliação online para um determinado agente e define uma taxa de amostragem e os avaliadores a serem aplicados para esta configuração. O AgentCore Evaluations então avaliará automaticamente o agente em produção, analisando os traces produzidos de acordo com a taxa de amostragem definida. O desenvolvedor pode então usar os dashboards do AgentCore Observability para visualizar os traces e as pontuações de avaliação do agente para atualizar continuamente o agente de acordo com os resultados das avaliações.


![session_traces](../../images/online_evaluations.png)

### Recuperando informações de tutoriais anteriores

Para este tutorial, usaremos o agente LangGraph implantado no AgentCore Runtime durante nosso tutorial de pré-requisitos. Vamos avaliá-lo com métricas pré-construídas e com a métrica `response_quality` que criamos no tutorial `01-creating-custom-metrics`. Vamos recuperar as informações do nosso agente e avaliador.

In [ ]:
%store -r launch_result_langgraph
%store -r evaluator_id
try:
    print("Agent Id:", launch_result_langgraph.agent_id)
    print("Agent ARN:", launch_result_langgraph.agent_arn)
except NameError as e:
    raise Exception("""Missing launch results from your LangGraph agent. Please run 00-prereqs before executing this lab""")

try:
    print("Evaluator id:", evaluator_id)
except NameError as e:
    raise Exception("""Missing custom evaluator id. Please run 01-creating-custom-evaluators before executing this lab""")

### Iniciando o cliente do AgentCore Evaluations

Agora vamos iniciar o cliente do AgentCore Evaluations a partir do AgentCore Starter toolkit.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation, Observability
import os
import json
from boto3.session import Session
from IPython.display import Markdown, display

In [ ]:
boto_session = Session()
region = boto_session.region_name
print(region)

In [ ]:
eval_client = Evaluation(region=region)

### Configurando a avaliação online

Vamos agora configurar a avaliação online. Neste caso, avaliaremos cada trace produzido, pois estamos usando nosso agente apenas para fins de demonstração. Em aplicações reais, você deve definir a taxa de amostragem de acordo com a utilização do seu agente.

Criaremos uma configuração de avaliação com as 5 métricas que exploramos na avaliação sob demanda:
* Builtin.GoalSuccessRate
* Builtin.Correctness
* Builtin.ToolParameterAccuracy
* Builtin.ToolSelectionAccuracy e
* nossa métrica personalizada: response_Quality

In [ ]:
response = eval_client.create_online_config(
    agent_id=launch_result_langgraph.agent_id,
    config_name="langgraph_agent_eval",
    sampling_rate=100,
    evaluator_list=[
        "Builtin.GoalSuccessRate", "Builtin.Correctness", 
        "Builtin.ToolParameterAccuracy", "Builtin.ToolSelectionAccuracy",
        evaluator_id
    ],
    config_description="LangGraph agent online evaluation test",
    auto_create_execution_role=True
)

### Analisando a configuração de avaliação

Vamos ver o ID de configuração da nossa configuração de avaliação online:

In [ ]:
print("Online Evaluation Configuration Id:", response['onlineEvaluationConfigId'])

Também podemos ver os detalhes da configuração criada para confirmar que ela já está habilitada:

In [ ]:
eval_client.get_online_config(config_id=response['onlineEvaluationConfigId'])

### Invocando o agente para acionar a avaliação

Vamos agora invocar nosso agente com algumas novas consultas para acionar nossa avaliação online. Desta vez, invocaremos nosso agente com boto3, pois uma vez que o endpoint está disponível, você pode invocá-lo a partir de qualquer interface.

In [ ]:
import boto3
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

def invoke_agent_runtime(agent_arn, prompt):
    boto3_response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        payload=json.dumps({"prompt": prompt})
    )
    if "text/event-stream" in boto3_response.get("contentType", ""):
        content = []
        for line in boto3_response["response"].iter_lines(chunk_size=1):
            if line:
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    line = line[6:]
                    print(line)
                    content.append(line)
        display(Markdown("\n".join(content)))
    else:
        try:
            events = []
            for event in boto3_response.get("response", []):
                events.append(event)
        except Exception as e:
            events = [f"Error reading EventStream: {e}"]
        display(Markdown(json.loads(events[0].decode("utf-8"))))
    return boto3_response


In [ ]:
response = invoke_agent_runtime(
    launch_result_langgraph.agent_arn,
    "How much is 7+9+10*2?"
)

In [ ]:
response = invoke_agent_runtime(
    launch_result_langgraph.agent_arn,
    "Is it raining?"
)

In [ ]:
response = invoke_agent_runtime(
    launch_result_langgraph.agent_arn,
    "how much is 20% of 300?"
)

In [ ]:
response = invoke_agent_runtime(
    launch_result_langgraph.agent_arn,
    "What can you do?"
)

In [ ]:
response = invoke_agent_runtime(
    launch_result_langgraph.agent_arn,
    "What is the capital of NY State?"
)

### Visualizando a avaliação online

Uma vez que você crie interações suficientes com seu agente, você pode usar o [console do AgentCore Observability](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents) para visualizar como ele está se saindo de acordo com sua configuração de avaliação online.

Navegue até o endpoint `DEFAULT` do seu agente para ver as avaliações atuais

**Importante**: Os resultados da avaliação podem demorar um pouco para aparecer no seu dashboard. Se o dashboard de avaliação estiver vazio, aguarde alguns minutos e verifique novamente.

Uma vez disponíveis, você poderá ver suas métricas diretamente nos traces do agente:
![image.png](../../images/online_evaluations_dashboard.png)

### Parabéns!

Você criou sua primeira configuração de avaliação online! Agora você pode criar métricas personalizadas e avaliar seu agente sob demanda e online com o AgentCore Evaluations!